# 05 — Inference & Deployment

**Pipeline:** Dataset → EDA → Preprocessing → Augmentation → Training → Evaluation → Error Analysis → **Inference → Deployment**

`Predictor` (in [`src/inference.py`](../src/inference.py)) loads `best.pt` once and predicts
from a path or a PIL image — the same class powers the CLI and the Gradio app.

In [ ]:
# run from notebooks/ -- make the repo root importable and the cwd
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from src.utils import load_config, seed_everything
cfg = load_config("configs/config.yaml")
seed_everything(cfg["seed"])
print("project:", cfg["project_name"], "| data root:", cfg["data"]["root"])

In [ ]:
from src.inference import Predictor

predictor = Predictor(cfg)
print("model ready on", predictor.device, "| classes:", predictor.classes)

## Predict on test images the model has never trained on

In [ ]:
from src.dataset import build_datasets
from PIL import Image
import matplotlib.pyplot as plt
import random

ds = build_datasets(cfg)
rng = random.Random(0)
picks = rng.sample(ds["split_indices"]["test"], 8)

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, idx in zip(axes.ravel(), picks):
    path, target = ds["samples"][idx]
    preds = predictor.predict(path, top_k=2)
    top = preds[0]
    ok = top["label"] == ds["classes"][target]
    ax.imshow(Image.open(path).convert("RGB")); ax.axis("off")
    ax.set_title(f"true: {ds['classes'][target]}\npred: {top['label']} ({top['confidence']:.2f})",
                 fontsize=9, color="green" if ok else "red")
plt.tight_layout(); plt.show()

## CLI

The same thing from a terminal — single image or a whole folder:

```bash
python -m src.inference --config configs/config.yaml --input path/to/image.jpg
python -m src.inference --config configs/config.yaml --input path/to/folder --top-k 3
```

## Deployment: the demo app

```bash
python app/app.py           # local  http://127.0.0.1:7860
python app/app.py --share   # temporary public gradio.live link
```

Drag an image in, get the class probabilities — this is the artifact you show people.
For free permanent hosting, push the repo + `outputs/checkpoints/best.pt` to a
**Hugging Face Space** (Gradio SDK): the Space runs `app/app.py` as-is.

## A confidence threshold for the real world

A classifier always answers, even on garbage input. In production, refuse to answer below a threshold — pick it from the confidence histogram in notebook 04:

In [ ]:
THRESHOLD = 0.7   # tune from notebook 04's histogram

def predict_or_abstain(image, threshold=THRESHOLD):
    top = predictor.predict(image, top_k=1)[0]
    if top["confidence"] < threshold:
        return {"label": "uncertain -- needs human review", **top}
    return top

path, _ = ds["samples"][ds["split_indices"]["test"][0]]
print(predict_or_abstain(path))

---

## You now have the full pipeline

Dataset → EDA → Preprocessing → Augmentation → Training → Evaluation → Error Analysis → Inference → Deployment

**To reuse this kit on a new dataset:** change `data.root` in the config, rerun notebooks 01→05. That's the whole point.

---
*Computer Vision Project Starter Kit — free, MIT-licensed.*
*By **Sofyan Tandungan** ([@sofyantandungan](https://instagram.com/sofyantandungan)) · NeuronEngine.*
*Built alongside the **Computer Vision with PyTorch** practical masterclass — 12 chapters, 100% runnable code, EN + ID.*